# Text Classification using LSTM and Word2Vec
## Architecture:
(X) Text -> Embedding (W2V pretrained on wikipedia articles) -> Deep Network (LSTM/GRU) -> Fully connected (Dense) -> Output Layer (Softmax) -> Emotion class (Y)

## Embedding Layer
Word Embedding is a representation of text where words that have the similar meaning have a similar representation. We will use 300 dimentional word vectors pre-trained on wikipedia articles. We can also train the w2v model with our data, however our dataset is quite small and trained word vectors might not be as good as using pretrained w2v.

## Deep Network
Deep network takes the sequence of embedding vectors as input and converts them to a compressed representation. The compressed representation effectively captures all the information in the sequence of words in the text. The deep network part is usually an RNN or some forms of it like LSTM/GRU. The dropout can be added to overcome the tendency to overfit, a very common problem with RNN based networks.

## Fully Connected Layer
The fully connected layer takes the deep representation from the RNN/LSTM/GRU and transforms it into the final output classes or class scores. This component is comprised of fully connected layers along with batch normalization and optionally dropout layers for regularization.

## Output Layer
Based on the problem at hand, this layer can have either Sigmoid for binary classification or Softmax for both binary and multi classification output.

## Workflow:
- Import Data
- Prepare the input data
- Import pre-trained W2V
- Create Neural Network Pipeline
- Train The Model
- Evaluate result

### 1. Import Data

In [ ]:
import pandas as pd
import numpy as np

# text preprocessing
from nltk.tokenize import word_tokenize
import re

# plots and metrics
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

# preparing input to our model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

# keras layers
from keras.models import Sequential
from keras.layers import Embedding, Bidirectional, LSTM, GRU, Dense, Dropout

In [ ]:
from sklearn.datasets import fetch_20newsgroups

train_data = fetch_20newsgroups(subset='train')
test_data = fetch_20newsgroups(subset='test')

X_train = train_data.data
X_test = test_data.data

y_train = train_data.target
y_test = test_data.target

print(train_data.data[0])
print('-'*50)
print(train_data.target_names[train_data.target[0]])

In [ ]:
print(len(train_data.data))
print(len(test_data.data))
print(len(train_data.target_names))

In [ ]:
train_data.target_names

In [ ]:
pd.Series(train_data.target).map(lambda x: train_data.target_names[x]).value_counts().plot(kind='barh')

In [ ]:
lengths = [len(text.split()) for text in X_train]

In [ ]:
np.percentile(lengths, 90) # this is why we will use max_seq_len = 500

Defining vector space dimension and fixed input size

In [ ]:
# Number of Labels
num_classes = 20

# Number of dimensions for word embedding
embed_num_dims = 300

# Max input length (max number of words)
max_seq_len = 500

class_names = train_data.target_names

## 2. Text Preprocessing

In [ ]:

import nltk
nltk.download('punkt_tab')
def clean_text(data):

    # lowercasing
    data = data.lower()

    # remove urls
    data = re.sub(r'http\S+', '', data)

    # remove hashtags and mentions
    data = re.sub(r"(#[\w\d_]+)", '', data)
    data = re.sub(r"(@[\w\d_]+)", '', data)

    # remove punctuation and numbers
    data = re.sub(r'[^a-z\s]', '', data)

    # tokenization
    data = word_tokenize(data)

    return data

In [ ]:
texts_train = [' '.join(clean_text(text)) for text in X_train]
texts_test = [' '.join(clean_text(text)) for text in X_test]

In [ ]:

tokenizer = Tokenizer()
tokenizer.fit_on_texts(texts_train)


sequence_train = tokenizer.texts_to_sequences(texts_train)
sequence_test = tokenizer.texts_to_sequences(texts_test)

index_of_words = tokenizer.word_index

# Vocab size is the number of unique words + 0 reserved to padding
vocab_size = len(index_of_words) + 1

print(f"Number of unique words: : {vocab_size}")


In [ ]:
X_train_pad = pad_sequences(sequence_train, maxlen = max_seq_len )
X_test_pad = pad_sequences(sequence_test, maxlen = max_seq_len )

X_train_pad

In [ ]:
y_train = to_categorical(y_train)
y_test = to_categorical(y_test)

y_train

## 3. Import pretrained word vectors
- Importing pretrained word2vec from file and creating embedding matrix
- We will later map each word in our corpus to existing word vector

In [ ]:
def create_embedding_matrix(filepath, word_index, embedding_dim):
    vocab_size = len(word_index) + 1  # Adding again 1 because of reserved 0 index
    embedding_matrix = np.zeros((vocab_size, embedding_dim))
    with open(filepath) as f:
        for line in f:
            word, *vector = line.split()
            if word in word_index:
                idx = word_index[word]
                embedding_matrix[idx] = np.array(
                    vector, dtype=np.float32)[:embedding_dim]
    return embedding_matrix

In [ ]:
import urllib.request
import zipfile
import os

fname = 'embeddings/wiki-news-300d-1M.vec'

if not os.path.isfile(fname):
    print('Downloading word vectors...')
    urllib.request.urlretrieve('https://dl.fbaipublicfiles.com/fasttext/vectors-english/wiki-news-300d-1M.vec.zip',
                              'wiki-news-300d-1M.vec.zip')
    print('Unzipping...')
    with zipfile.ZipFile('wiki-news-300d-1M.vec.zip', 'r') as zip_ref:
        zip_ref.extractall('embeddings')
    print('done.')

    os.remove('wiki-news-300d-1M.vec.zip')

In [ ]:
embedded_matrix = create_embedding_matrix(fname, index_of_words, embedding_dim=300)
print(embedded_matrix.shape)

In [ ]:
# Inspect unseen words
new_words = 0

for word in index_of_words:
    entry = embedded_matrix[index_of_words[word]]
    if all(v == 0 for v in entry):
        new_words = new_words + 1

print('Words found in wiki vocab: ' + str(len(index_of_words) - new_words))
print('New words found: ' + str(new_words))

# 3. Create LSTM Pipeline
## Embedding Layer
We will use pre-trained word vectors. We could also train our own embedding layer if we don't specify the pre-trained weights

- **vocabulary size**: the maximum number of terms that are used to represent a text: e.g. if we set the size of the “vocabulary” to 1000 only the first thousand terms most frequent in the corpus will be considered (and the other terms will be ignored)

- **the maximum length**: of the texts (which must all be the same length)

- **size of embeddings**: basically, the more dimensions we have the more
precise the semantics will be, but beyond a certain threshold we will lose the ability of the embedding to define a coherent and general enough semantic area

- **trainable**: True if you want to fine-tune them while training

In [ ]:

embedd_layer = Embedding(
    vocab_size,
    embed_num_dims,
    input_length = max_seq_len,
    weights = [embedded_matrix],
    trainable = False
)

## Model Pipeline
- **the input** is the first N words of each text (with proper padding)

- **the first level** creates embedding of words, using vocabulary with a certain dimension, and a given size of embeddings

- **LSTM/GRU layer** which will receive word embeddings for each token in the tweet as inputs. The intuition is that its output tokens will store information not only of the initial token, but also any previous tokens; In other words, the -
- **LSTM layer** is generating a new encoding for the original input.

- **the output level** has a number of neurons equal to the classes of the problem and a “softmax” activation function You can change GRU to LSTM. The results will be very similar but LSTM might take longer to train.

In [ ]:
# Parameters
gru_output_size = 128
bidirectional = True

# Embedding Layer, LSTM or biLSTM, Dense, softmax
model = Sequential()
model.add(embedd_layer)

if bidirectional:
    model.add(Bidirectional(GRU(units=gru_output_size,
                              dropout=0.3,
                              recurrent_dropout=0.2)))
else:
     model.add(GRU(units=gru_output_size,
                dropout=0.3,
                recurrent_dropout=0.2))

model.add(Dense(64, activation='relu'))
model.add(Dropout(0.4))
model.add(Dense(num_classes, activation='softmax'))

In [ ]:

model.compile(loss = 'categorical_crossentropy', optimizer = 'adam', metrics = ['accuracy'])
model.build(input_shape=(None, max_seq_len))
model.summary()


In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.callbacks import EarlyStopping

checkpoint = ModelCheckpoint(
    "best_model.keras",
    monitor="val_loss",
    save_best_only=True,
    mode="min",
    verbose=1
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

In [ ]:
batch_size = 64
epochs = 20

hist = model.fit(X_train_pad, y_train,
                 batch_size=batch_size,
                 epochs=epochs,
                 validation_data=(X_test_pad,y_test),
                 callbacks=[checkpoint, early_stop]
                 )

In [ ]:
predictions = model.predict(X_test_pad)
predictions = np.argmax(predictions, axis=1)
predictions = [test_data.target_names[pred] for pred in predictions]

In [ ]:
y_true = [test_data.target_names[i] for i in test_data.target]

print("Accuracy: {:.2f}%".format(accuracy_score(y_true, predictions) * 100))
print("\nF1 Score: {:.2f}".format(f1_score(y_true , predictions, average='micro') * 100))

In [ ]:
def plot_confusion_matrix(y_true, y_pred, classes,
                          normalize=False,
                          title=None,
                          cmap=plt.cm.Blues):

    if not title:
        if normalize:
            title = 'Normalized confusion matrix'
        else:
            title = 'Confusion matrix, without normalization'

    # Compute confusion matrix
    cm = confusion_matrix(y_true, y_pred)

    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    fig, ax = plt.subplots(figsize=(18, 14))

    im = ax.imshow(cm, interpolation='nearest', cmap=cmap)
    ax.figure.colorbar(im, ax=ax)
    ax.grid(False)

    ax.set(
        xticks=np.arange(cm.shape[1]),
        yticks=np.arange(cm.shape[0]),
        xticklabels=classes,
        yticklabels=classes,
        title=title,
        ylabel='True label',
        xlabel='Predicted label'
    )


    plt.setp(ax.get_xticklabels(), rotation=90, ha="right", fontsize=10)
    plt.setp(ax.get_yticklabels(), fontsize=10)

    # Annotate
    fmt = '.2f' if normalize else 'd'
    thresh = cm.max() / 2.

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i,
                    format(cm[i, j], fmt),
                    ha="center",
                    va="center",
                    fontsize=8,
                    color="white" if cm[i, j] > thresh else "black")

    fig.tight_layout()
    return ax

In [ ]:
#  "Accuracy"
plt.plot(hist.history['accuracy'])
plt.plot(hist.history['val_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'validation'], loc='upper left')
plt.show()

# "Loss"
plt.plot(hist.history['loss'])
plt.plot(hist.history['val_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'validation'], loc='upper left')
plt.show()

In [ ]:
print("\nF1 Score: {:.2f}".format(f1_score(y_true , predictions, average='micro') * 100))

# Plot normalized confusion matrix
plot_confusion_matrix(y_true , predictions, classes=class_names, normalize=True, title='Normalized confusion matrix')
plt.show()